# IV. Multivariate Analysis

This notebook evaluates campaign performance using multivariate techniques on the cleaned dataset.
The goal is to understand how multiple features interact and jointly explain ROI and conversion behavior.

## Step 1: Imports and Visualization Setup

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.figsize"] = (12, 6)
pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 140)

## Step 2: Load Cleaned Dataset

In [ ]:
cleaned_dataset_path = Path("../dataset/marketing_campaign_dataset_cleaned.csv")

if not cleaned_dataset_path.exists():
    raise FileNotFoundError(
        f"Cleaned dataset not found at: {cleaned_dataset_path.resolve()}\n"
        "Run 02_clean_and_validate.ipynb first to generate the cleaned data."
    )

df = pd.read_csv(cleaned_dataset_path, parse_dates=["Date"])

print(f"Loaded dataframe shape: {df.shape}")
print(f"Source file: {cleaned_dataset_path.resolve()}")
df.head()

## Step 3: Build Multivariate Modeling Dataset

Prepare numeric predictors and encoded categorical signals for multivariate analysis.

In [ ]:
target = "ROI"

numeric_features = [
    "Conversion_Rate",
    "Acquisition_Cost",
    "Clicks",
    "Impressions",
    "Engagement_Score",
    "Duration",
]

categorical_features = ["Channel_Used", "Campaign_Type", "Target_Audience", "Customer_Segment"]

required = [target, *numeric_features, *categorical_features]
missing_required = [c for c in required if c not in df.columns]
if missing_required:
    raise ValueError(f"Missing required columns for multivariate analysis: {missing_required}")

model_df = df[required].copy()
model_df = model_df.dropna().reset_index(drop=True)

X_num = model_df[numeric_features].copy()
X_cat = pd.get_dummies(model_df[categorical_features], drop_first=True)
X = pd.concat([X_num, X_cat], axis=1)
y = model_df[target].astype(float)

print(f"Rows used for modeling: {len(model_df)}")
print(f"Total predictors after encoding: {X.shape[1]}")
X.head()

## Step 4: Multivariate Visualization (Pair Relationships)

A pairplot helps visualize relationships among several numeric variables simultaneously.

In [ ]:
pair_cols = ["ROI", "Conversion_Rate", "Acquisition_Cost", "Clicks", "Impressions", "Engagement_Score"]
pair_df = model_df[pair_cols].sample(min(3000, len(model_df)), random_state=42)

sns.pairplot(pair_df, corner=True, diag_kind="hist", plot_kws={"alpha": 0.3, "s": 12})
plt.suptitle("Pairplot of Key Numeric Variables (Sample)", y=1.02)
plt.show()

## Step 5: Multiple Linear Regression for ROI

Fit a multivariate regression model where ROI is explained by multiple numeric and encoded categorical predictors.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)

reg = LinearRegression()
reg.fit(X_train, y_train)

y_pred = reg.predict(X_test)

r2 = r2_score(y_test, y_pred)
mae = mean_absolute_error(y_test, y_pred)
rmse = mean_squared_error(y_test, y_pred, squared=False)

print(f"R^2:  {r2:.4f}")
print(f"MAE:  {mae:.4f}")
print(f"RMSE: {rmse:.4f}")

coef_df = pd.DataFrame({"feature": X.columns, "coefficient": reg.coef_})
coef_df["abs_coefficient"] = coef_df["coefficient"].abs()
coef_df = coef_df.sort_values("abs_coefficient", ascending=False)

print("Top 15 coefficient magnitudes:")
coef_df.head(15)

In [ ]:
plt.figure(figsize=(8, 6))
sns.scatterplot(x=y_test, y=y_pred, alpha=0.4)
min_v = min(y_test.min(), y_pred.min())
max_v = max(y_test.max(), y_pred.max())
plt.plot([min_v, max_v], [min_v, max_v], color="red", linestyle="--", linewidth=1.5)
plt.title("Actual vs Predicted ROI (Multivariate Regression)")
plt.xlabel("Actual ROI")
plt.ylabel("Predicted ROI")
plt.tight_layout()
plt.show()

## Step 6: Customer/Campaign Segmentation with K-Means

Cluster campaigns on core performance metrics to identify multivariate behavioral segments.

In [ ]:
cluster_features = ["Conversion_Rate", "Acquisition_Cost", "ROI", "Clicks", "Impressions", "Engagement_Score"]
cluster_df = model_df[cluster_features].dropna().copy()

scaler = StandardScaler()
X_cluster_scaled = scaler.fit_transform(cluster_df)

# Try a few K values and choose the one with best silhouette score.
candidate_k = [3, 4, 5, 6]
scores = []
for k in candidate_k:
    km = KMeans(n_clusters=k, n_init=20, random_state=42)
    labels = km.fit_predict(X_cluster_scaled)
    score = silhouette_score(X_cluster_scaled, labels)
    scores.append((k, score))

silhouette_df = pd.DataFrame(scores, columns=["k", "silhouette_score"]).sort_values("silhouette_score", ascending=False)
best_k = int(silhouette_df.iloc[0]["k"])

print("Silhouette scores by K:")
display(silhouette_df)
print(f"Selected K: {best_k}")

kmeans = KMeans(n_clusters=best_k, n_init=20, random_state=42)
cluster_df["cluster"] = kmeans.fit_predict(X_cluster_scaled)
cluster_df.head()

In [ ]:
cluster_profile = cluster_df.groupby("cluster").agg(["mean", "median", "std", "count"])
cluster_profile

In [ ]:
pca = PCA(n_components=2, random_state=42)
pcs = pca.fit_transform(X_cluster_scaled)

pca_plot_df = pd.DataFrame({
    "PC1": pcs[:, 0],
    "PC2": pcs[:, 1],
    "cluster": cluster_df["cluster"].values,
})

plt.figure(figsize=(10, 7))
sns.scatterplot(data=pca_plot_df, x="PC1", y="PC2", hue="cluster", palette="tab10", alpha=0.65)
plt.title("K-Means Clusters Projected onto 2D PCA Space")
plt.tight_layout()
plt.show()

explained = pca.explained_variance_ratio_
print(f"Explained variance ratio - PC1: {explained[0]:.4f}, PC2: {explained[1]:.4f}")

## Step 7: Multivariate Insights Template

Use results above to summarize:
- Which variables jointly contribute most to ROI in the regression model?
- How strong is predictive fit (R^2, MAE, RMSE) and what does that imply?
- How many meaningful campaign clusters were identified and how are they different?
- Which cluster profile should be prioritized for budget allocation?

## Summary

This notebook completed multivariate analysis on the cleaned dataset using:
- simultaneous numeric relationship inspection
- multiple linear regression for ROI
- unsupervised clustering and PCA-based cluster visualization.

These outputs support higher-confidence decisions on channel mix, audience strategy, and campaign segmentation.